# Comparación gráfica A5: SciBERT base, Luna y Gemini

Genera una figura de F1 macro y F1 micro a partir de `comparacion_metricas.csv` ya validado.
Incluye SciBERT base y las cuatro condiciones de cada comercial sobre los mismos 2.968 casos.
No entrena modelos ni utiliza claves o APIs. Los originales permanecen sin cambios.

## 1. Bibliotecas y ubicación del CSV

Localiza la tabla generada por `comparar_modelos.py`. Si se utiliza Colab, copia primero
`comparacion_metricas.csv` a tu Drive y ajusta `RESULTS_DIR`. Solo requiere Matplotlib y Python.

In [ ]:
from pathlib import Path
import csv
import hashlib
import json
import math
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

CANDIDATES = [
    Path.cwd() / "Comparacion_SciBERT/resultados",
    Path.cwd() / "A5/Comparacion_SciBERT/resultados",
    Path.cwd() / "resultados",
    Path("/content/drive/MyDrive/MAIA_TESIS/A5/Comparacion_SciBERT/resultados"),
]
RESULTS_DIR = next((p for p in CANDIDATES if (p / "comparacion_metricas.csv").is_file()), CANDIDATES[0])
CSV_PATH = RESULTS_DIR / "comparacion_metricas.csv"
assert CSV_PATH.is_file(), "Configura RESULTS_DIR con la carpeta que contiene comparacion_metricas.csv"
print("Tabla de entrada:", CSV_PATH)

## 2. Leer y comprobar las nueve configuraciones

Exige un resultado por configuración, 2.968 casos por fila y valores F1 entre 0 y 1.
Las respuestas inválidas ya están incluidas como errores en las métricas de origen.

In [ ]:
with CSV_PATH.open(encoding="utf-8-sig", newline="") as stream:
    rows = list(csv.DictReader(stream))
ORDER = [("SciBERT base", "base")] + [
    (model, condition)
    for model in ("gpt-6-luna", "gemini-3.5-flash-lite")
    for condition in ("base-generic-zero", "base-generic-few", "base-specific-zero", "base-specific-few")
]
lookup = {(r["model"], r["condition"]): r for r in rows}
assert len(rows) == len(lookup) == 9 and set(lookup) == set(ORDER)
rows = [lookup[key] for key in ORDER]
for row in rows:
    assert int(row["n"]) == 2968
    for metric in ("f1_macro", "f1_micro"):
        value = float(row[metric])
        assert math.isfinite(value) and 0 <= value <= 1
print("Verificado: nueve configuraciones, 2.968 casos por configuración.")

## 3. Construir la figura comparativa

Las barras usan una escala común de 0 a 1. Los colores identifican la familia del modelo.
La línea discontinua señala el valor de SciBERT en cada métrica. Es una comparación descriptiva;
los intervalos pareados del análisis estadístico están en la tabla y el informe, no en estas barras.

In [ ]:
FAMILY = {"SciBERT base": "#167D8D", "gpt-6-luna": "#6554A4", "gemini-3.5-flash-lite": "#C26A17"}
CONDITION_LABELS = {
    "base-generic-zero": "Genérico · sin ejemplos",
    "base-generic-few": "Genérico · con ejemplos",
    "base-specific-zero": "Específico · sin ejemplos",
    "base-specific-few": "Específico · con ejemplos",
}
labels = ["SciBERT base"] + [
    ("Luna" if row["model"] == "gpt-6-luna" else "Gemini Flash-Lite") + " · " + CONDITION_LABELS[row["condition"]]
    for row in rows[1:]
]
colors = [FAMILY[row["model"]] for row in rows]
plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 11, "pdf.fonttype": 42})
fig, axes = plt.subplots(1, 2, figsize=(16, 8.2), sharey=True)
y = list(range(len(rows)))
for ax, metric, title in zip(axes, ("f1_macro", "f1_micro"), ("F1 macro", "F1 micro")):
    values = [float(row[metric]) for row in rows]
    ax.barh(y, values, color=colors, height=.64, zorder=3)
    ax.axvline(values[0], color=FAMILY["SciBERT base"], linestyle="--", linewidth=1.4, alpha=.8, zorder=2)
    for pos, value in zip(y, values):
        ax.text(value+.012, pos, f"{value:.4f}", va="center", fontsize=10)
    for boundary in (.5, 4.5):
        ax.axhline(boundary, color="#D9DFE6", linewidth=1)
    ax.set_xlim(0, 1)
    ax.set_xticks([i/10 for i in range(11)])
    ax.set_xlabel("Puntaje F1 (0–1)")
    ax.set_title(title, fontsize=16, fontweight="bold", pad=14)
    ax.grid(axis="x", color="#E7EBF0", linewidth=.8, zorder=0)
    ax.set_axisbelow(True)
    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.tick_params(axis="y", length=0)
axes[0].set_yticks(y, labels=labels)
axes[0].invert_yaxis()
fig.suptitle("Funciones de cita · comparación final en T_B", fontsize=21, fontweight="bold", y=.965)
fig.legend(handles=[Patch(facecolor=color, label=name) for name, color in [
    ("SciBERT base", FAMILY["SciBERT base"]), ("GPT-6 Luna", FAMILY["gpt-6-luna"]),
    ("Gemini 3.5 Flash-Lite", FAMILY["gemini-3.5-flash-lite"]) ]],
    loc="upper center", bbox_to_anchor=(.63, .91), ncol=3, frameon=False)
fig.text(.03, .035, "2.968 casos por configuración · mismas etiquetas humanas · respuestas inválidas incluidas como errores", fontsize=10, color="#465161")
fig.text(.03, .012, "Barras descriptivas; una diferencia pequeña no demuestra superioridad ni equivalencia. Línea discontinua: referencia SciBERT.", fontsize=9, color="#465161")
fig.subplots_adjust(left=.34, right=.96, bottom=.13, top=.80, wspace=.17)

## 4. Exportar PNG y PDF

Guarda un PNG de 300 dpi y un PDF vectorial en la carpeta de resultados. Añade un manifiesto
con el hash de la tabla utilizada y de ambas figuras para identificar su procedencia.

In [ ]:
PNG_PATH = RESULTS_DIR / "Comparacion_Metricas_A5.png"
PDF_PATH = RESULTS_DIR / "Comparacion_Metricas_A5.pdf"
fig.savefig(PNG_PATH, dpi=300, facecolor="white", bbox_inches="tight")
fig.savefig(PDF_PATH, facecolor="white", bbox_inches="tight")
manifest = {
    "source_csv": CSV_PATH.name,
    "source_sha256": hashlib.sha256(CSV_PATH.read_bytes()).hexdigest(),
    "cases_per_configuration": 2968,
    "configurations": 9,
    "metrics": ["f1_macro", "f1_micro"],
    "axis_limits": [0, 1],
    "files": {p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in (PNG_PATH, PDF_PATH)},
    "api_called": False,
}
(RESULTS_DIR / "figura_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2)+"\n", encoding="utf-8")
print("PNG:", PNG_PATH)
print("PDF:", PDF_PATH)

## 5. Mostrar la figura

SciBERT es una sola referencia supervisada. Los comerciales tienen cuatro estrategias de prompting;
SciBERT no se interpreta como un experimento zero-shot o few-shot.

In [ ]:
plt.show()